In [ ]:


import numpy as np
import pickle
import tensorflow.compat.v1 as tf
import random, math, os
from matplotlib import pyplot as plt

tf.disable_v2_behavior()
random.seed(42); np.random.seed(42); tf.set_random_seed(42)


xdim    = 5
T_final = 1.0
dt_hjb  = 0.01
layers  = [128, 128, 64, 1]

address = "/content/drive/MyDrive/table 1 (2), case 1/0.6"
os.makedirs(address, exist_ok=True)


try:
    with open(address + '/zinitial_r1.data', 'rb') as fm0:
        x0_all = pickle.load(fm0)
    with open(address + '/z_generated_A_r1.data', 'rb') as fm00A:
        A = pickle.load(fm00A)
    with open(address + '/z_errcheck_r1.data', 'rb') as fm00:
        x_phierr = pickle.load(fm00)
    print("Loaded existing data")
except Exception as e:
    print("Generating new data due to:", e)
    x0_all = np.random.uniform(-0.5, 0.5, size=(250, 20, xdim))
    a = np.random.randn(xdim, xdim) * 0.05
    A_temp = (np.eye(xdim))
    eigvals = np.linalg.eigvals(A_temp)
    max_real = np.max(np.real(eigvals))
    if max_real > 0:
        A = A_temp - (max_real + 0.5) * np.eye(xdim)
    else:
        A = A_temp - 0.5 * np.eye(xdim)
    x_phierr = np.random.uniform(-0.5, 0.5, size=(10000, xdim))
    with open(address + '/zinitial_r1.data', 'wb') as fm:
        pickle.dump(x0_all, fm)
    with open(address + '/z_generated_A_r1.data', 'wb') as fm2:
        pickle.dump(A, fm2)
    with open(address + '/z_errcheck_r1.data', 'wb') as fm1:
        pickle.dump(x_phierr, fm1)

A = np.array(A, dtype=np.float32)
B = np.eye(xdim, dtype=np.float32)

A_tf_const = tf.constant(A, dtype=tf.float32)
B_tf_const = tf.constant(B, dtype=tf.float32)


def terminal_cost(x_term):
    return 0.5 * np.sum(x_term**2)

def terminal_gradient(x_term):
    return x_term


x = tf.placeholder("float32", [None, xdim])
t = tf.placeholder("float32", [None, 1])
y_ = tf.placeholder("float32", [None, 1])
lambda_target = tf.placeholder("float32", [None, xdim])
dv_dt_target = tf.placeholder("float32", [None, 1])
u_opt_placeholder = tf.placeholder("float32", [None, xdim])

value_weight    = tf.placeholder("float32")
gradient_weight = tf.placeholder("float32")
pde_weight      = tf.placeholder("float32")
keep_prob       = tf.placeholder("float32")
is_training     = tf.placeholder(tf.bool)


def dense_layer(inputs, units, activation=None, name=None):
    input_dim = inputs.get_shape().as_list()[1]
    with tf.variable_scope(name or "dense"):
        W = tf.get_variable("W", [input_dim, units],
                            initializer=tf.keras.initializers.GlorotUniform())
        b = tf.get_variable("b", [units],
                            initializer=tf.zeros_initializer())
        z = tf.matmul(inputs, W) + b
        return activation(z) if activation is not None else z

def neural_net(x_input, t_input, layers, name="value_net"):
    with tf.variable_scope(name, reuse=tf.AUTO_REUSE):
        net = tf.concat([x_input, t_input], axis=1)
        for i, width in enumerate(layers[:-1]):
            net = dense_layer(net, width, activation=tf.nn.tanh,
                              name=f"layer_{i}")
        return dense_layer(net, layers[-1], activation=None, name="output")

with tf.variable_scope('value_function'):
    y = neural_net(x, t, layers)
    lambda_approx = tf.gradients(y, x)[0]  # λ(x,t) = D_x V(x,t)


value_loss    = tf.reduce_mean(tf.square(y_ - y))
gradient_loss = tf.reduce_mean(tf.square(lambda_target - lambda_approx))

Ax = tf.matmul(x, tf.transpose(A_tf_const))
Bu = tf.matmul(u_opt_placeholder, tf.transpose(B_tf_const))
state_dynamics = Ax + Bu
grad_v = lambda_approx

running_cost = 0.5 * (
    tf.reduce_sum(x * x, axis=1, keepdims=True) +
    tf.reduce_sum(u_opt_placeholder * u_opt_placeholder,
                  axis=1, keepdims=True)
)


H_graph = tf.reduce_sum(grad_v * state_dynamics, axis=1, keepdims=True) \
          + running_cost


pde_loss = tf.reduce_mean(tf.square(-dv_dt_target - H_graph))

Loss = value_weight * value_loss \
       + gradient_weight * gradient_loss \
       + pde_weight * pde_loss

global_step = tf.train.get_or_create_global_step()
learning_rate = tf.train.exponential_decay(
    0.01, global_step, 500, 0.95, staircase=True
)
optimizer  = tf.train.AdamOptimizer(learning_rate)
train_step = optimizer.minimize(Loss, global_step=global_step)

sess = tf.Session()
sess.run(tf.global_variables_initializer())


def func1(xn, a, A, B):
    xn = xn.reshape(xdim, 1)
    a  = a.reshape(xdim, 1)
    return (np.matmul(A, xn) + np.matmul(B, a)).reshape(xdim)

def func2(xn, a):
    xn = xn.reshape(xdim)
    a  = a.reshape(xdim)
    return 0.5 * (np.linalg.norm(xn)**2 + np.linalg.norm(a)**2)


def make_trajectory_backward(t0, x0, A, B, dt=0.01, force_zero=True):

    T_list, X_list, V_list = [], [], []
    t_current = float(t0)
    x_current = np.array(x0).reshape(xdim)
    v_current = terminal_cost(x_current) if abs(t_current - T_final) < 1e-6 else 0.0

    while t_current > dt:
        a_current = np.zeros(xdim, dtype=np.float32) if force_zero \
            else -np.dot(B.T, solve_lambda(x_current, t_current))
        dt_actual = min(dt, t_current)
        dx_dt     = func1(x_current, a_current, A, B)
        x_prev    = x_current - dt_actual * dx_dt
        cost      = func2(x_prev, a_current)
        v_current += dt_actual * cost
        t_current -= dt_actual
        x_current  = x_prev
        if np.linalg.norm(x_current) > 10:
            break
        T_list.append(t_current)
        X_list.append(x_current.copy())
        V_list.append(v_current)

    T_list.reverse(); X_list.reverse(); V_list.reverse()
    return T_list, X_list, V_list


def integrate_lambda_along_trajectory(X_traj, A, dt=0.01, terminal_grad=None):

    n = len(X_traj)
    lambdas = [np.zeros_like(X_traj[0]) for _ in range(n)]
    if terminal_grad is None:
        terminal_grad = np.zeros_like(X_traj[-1])
    lambdas[-1] = np.array(terminal_grad, dtype=np.float32)

    def lam_dot(xk, lamk):
        grad_l = xk
        return -A.T.dot(lamk) - grad_l

    for k in range(n - 2, -1, -1):
        x_next, lam_next = X_traj[k + 1], lambdas[k + 1]
        x_curr = X_traj[k]
        h      = dt
        k1 = lam_dot(x_next, lam_next)
        k2 = lam_dot(0.5 * (x_next + x_curr), lam_next - 0.5 * h * k1)
        k3 = lam_dot(0.5 * (x_next + x_curr), lam_next - 0.5 * h * k2)
        k4 = lam_dot(x_curr, lam_next - h * k3)
        lambdas[k] = lam_next - (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return np.array(lambdas, dtype=np.float32)


def solve_lambda(x_val, t_val):
    grad = sess.run(lambda_approx, feed_dict={
        x: x_val.reshape(1, -1),
        t: np.array([[t_val]], dtype=np.float32),
        keep_prob: 1.0,
        is_training: False
    })
    return grad[0]

def solve_a(x_val, t_val, force_zero=True):
    if force_zero:
        return np.zeros(xdim, dtype=np.float32)
    lam = solve_lambda(x_val, t_val)
    return -np.dot(B.T, lam)


def heat_half_step_mc(epsilon, dt_half, x_points, u_prev, n_mc=16):

    if epsilon <= 0.0 or dt_half <= 0.0:
        return u_prev.copy()

    x_points = x_points.astype(np.float32)
    u_prev   = u_prev.astype(np.float32)
    N, d     = x_points.shape
    sigma    = math.sqrt(2.0 * epsilon * dt_half)

    Z = np.random.randn(N, n_mc, d).astype(np.float32)
    Y = x_points[:, None, :] + sigma * Z  # (N, n_mc, d)
    Y_flat = Y.reshape(-1, d)

    diff  = Y_flat[:, None, :] - x_points[None, :, :]  # (N*n_mc, N, d)
    dist2 = np.sum(diff**2, axis=2)
    idx   = np.argmin(dist2, axis=1)
    u_y   = u_prev[idx].reshape(N, n_mc, 1)

    u_new = np.mean(u_y, axis=1)
    return u_new.astype(np.float32)


def hj_half_step_using_nn(x_points, u_in, dt_half):

    if dt_half <= 0.0:
        return u_in.copy()

    x_points = x_points.astype(np.float32)
    u_in     = u_in.astype(np.float32)
    N        = x_points.shape[0]

    t_batch = np.zeros((N, 1), dtype=np.float32)

    lambda_vals = sess.run(lambda_approx, feed_dict={
        x: x_points,
        t: t_batch,
        keep_prob: 1.0,
        is_training: False
    })  # (N,d)

    Ax_np = np.dot(x_points, A.T)
    running_cost_np = 0.5 * np.sum(x_points**2, axis=1, keepdims=True)

    H_vals = np.sum(lambda_vals * Ax_np, axis=1, keepdims=True) \
             + running_cost_np

    u_out = u_in - dt_half * H_vals
    return u_out.astype(np.float32)


force_zero_global = True

def pi_lambda_update_and_residual(eta, tranum, itenum_pi=20,
                                  n_epochs_first=2000, n_epochs_rest=1000):

    mu = 1.0 - eta
    loss_residual = []

    for iter_idx in range(itenum_pi):
        if iter_idx % 1 == 0:
            print(f"    [PI-λ] Outer iter {iter_idx}")

        x_all, t_all, phi_all, lambda_all = [], [], [], []


        for j in range(tranum):
            x0 = x0_all[np.random.randint(x0_all.shape[0]),
                        j % x0_all.shape[1], :] * 0.3
            for t0 in np.linspace(0.1, T_final, 4):
                T_traj, X_traj, V_traj = make_trajectory_backward(
                    t0, x0, A, B, dt=0.01,
                    force_zero=True
                )
                if len(T_traj) > 5:
                    term_grad  = terminal_gradient(X_traj[-1])
                    LAMBDA_true = integrate_lambda_along_trajectory(
                        X_traj, A, dt=0.01, terminal_grad=term_grad
                    )
                    idxs = np.linspace(
                        0, len(T_traj)-1,
                        min(10, len(T_traj)),
                        dtype=int
                    )
                    for idx in idxs:
                        x_all.append(X_traj[idx])
                        t_all.append(T_traj[idx])
                        phi_all.append(V_traj[idx])
                        lambda_all.append(LAMBDA_true[idx])


        for _ in range(max(100, tranum * 5)):
            x_term = np.random.uniform(-0.3, 0.3, size=xdim)
            x_all.append(x_term)
            t_all.append(T_final)
            phi_all.append(terminal_cost(x_term))
            lambda_all.append(terminal_gradient(x_term))

        x_train      = np.array(x_all, dtype=np.float32)
        t_train      = np.array(t_all, dtype=np.float32).reshape(-1,1)
        y_train      = np.array(phi_all, dtype=np.float32).reshape(-1,1)
        lambda_train = np.array(lambda_all, dtype=np.float32)
        n_train      = len(x_train)

        batch_size = 64
        n_epochs   = n_epochs_first if iter_idx == 0 else n_epochs_rest
        best_loss  = float('inf')
        patience   = 20
        patience_counter = 0


        for epoch in range(n_epochs):
            indices = np.random.permutation(n_train)
            epoch_loss, n_batches = 0.0, 0

            for start in range(0, n_train, batch_size):
                end = min(start + batch_size, n_train)
                batch_idx = indices[start:end]

                t_batch = t_train[batch_idx]
                t_plus  = np.minimum(t_batch + dt_hjb, T_final).astype(np.float32)

                v_t = sess.run(y, feed_dict={
                    x: x_train[batch_idx],
                    t: t_batch,
                    keep_prob: 1.0,
                    is_training: False
                })
                v_t_dt = sess.run(y, feed_dict={
                    x: x_train[batch_idx],
                    t: t_plus,
                    keep_prob: 1.0,
                    is_training: False
                })
                dv_dt = (v_t_dt - v_t) / dt_hjb

                u_opt_batch = np.zeros((len(batch_idx), xdim), dtype=np.float32)

                feed = {
                    x: x_train[batch_idx],
                    t: t_batch,
                    y_: y_train[batch_idx],
                    lambda_target: lambda_train[batch_idx],
                    dv_dt_target: dv_dt,
                    u_opt_placeholder: u_opt_batch,
                    value_weight: eta,
                    gradient_weight: mu,
                    pde_weight: 20.0,
                    keep_prob: 0.95,
                    is_training: True
                }

                _, loss_val = sess.run([train_step, Loss], feed_dict=feed)
                epoch_loss += float(loss_val)
                n_batches  += 1

            avg_epoch_loss = epoch_loss / max(1, n_batches)
            if avg_epoch_loss < best_loss:
                best_loss = avg_epoch_loss
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter > patience:
                break


        residuals = []
        for _ in range(1000):
            x_test = x_phierr[np.random.randint(len(x_phierr))] * 0.3
            t_test = np.random.uniform(0.1, T_final - dt_hjb)

            v_t_val, grad_t_val = sess.run([y, lambda_approx], feed_dict={
                x: x_test.reshape(1, -1),
                t: np.array([[t_test]], dtype=np.float32),
                keep_prob: 1.0,
                is_training: False
            })
            v_t_val = v_t_val[0,0]
            grad_vec = grad_t_val[0]

            v_t_dt_val = sess.run(y, feed_dict={
                x: x_test.reshape(1, -1),
                t: np.array([[t_test + dt_hjb]], dtype=np.float32),
                keep_prob: 1.0,
                is_training: False
            })[0,0]
            dv_dt_val = (v_t_dt_val - v_t_val) / dt_hjb

            H_val = np.dot(grad_vec, np.dot(A, x_test)) \
                    + 0.5 * np.linalg.norm(x_test)**2
            residuals.append(abs(-dv_dt_val - H_val))

        avg_res = float(np.mean(residuals))
        loss_residual.append(avg_res)
        print(f"    [PI-λ] HJB Residual (avg over 1000 pts): {avg_res:.4e}")

    return loss_residual  # list over outer iters


def run_splitting_scheme_with_pi_lambda(epsilon, T_scheme, N_time,
                                        eta, tranum,
                                        itenum_pi=3, n_mc_heat=16):

    x_points = np.random.uniform(-0.5, 0.5,
                                 size=(300, xdim)
                                 ).astype(np.float32)
    u_prev = 0.5 * np.sum(x_points**2, axis=1, keepdims=True).astype(np.float32)

    dt      = T_scheme / float(N_time)
    dt_half = 0.5 * dt

    hjb_residuals_per_time = []

    for i in range(N_time):
        print(f"\n=== Scheme step i = {i}, time in [{i*dt:.3f}, {(i+1)*dt:.3f}] ===")

        u_heat = heat_half_step_mc(epsilon, dt_half, x_points,
                                   u_prev, n_mc=n_mc_heat)


        print("  Step – PI–λ update and HJB residual:")
        residual_list = pi_lambda_update_and_residual(
            eta=eta,
            tranum=tranum,
            itenum_pi=itenum_pi
        )
        hjb_residuals_per_time.append(residual_list)


        u_next = hj_half_step_using_nn(x_points, u_heat, dt_half)
        u_prev = u_next

    return x_points, u_prev, hjb_residuals_per_time

if __name__ == "__main__":
    epsilon   = 0.01
    T_scheme  = 2.0
    N_time    = 20
    eta       = 0.6
    tranum    = 12
    itenum_pi = 5

    x_pts, u_TN, hjb_res_all = run_splitting_scheme_with_pi_lambda(
        epsilon=epsilon,
        T_scheme=T_scheme,
        N_time=N_time,
        eta=eta,
        tranum=tranum,
        itenum_pi=itenum_pi,
        n_mc_heat=16
    )

    min_res_per_step = [min(res_list) for res_list in hjb_res_all]
    print("\n=== Summary over Schemes steps ")
    print("step i | min HJB residual during PI–λ at this step")
    print("-----------------------------------------------")
    for i, r in enumerate(min_res_per_step):
        print(f"{i:6d} | {r:.4e}")

    sess.close()
    print("\nDone.")
